# HTR Gratuito - Celorico da Beira (Churro-3B / Transkribus)Processa 4152 imagens de registos de óbitos via modelos open-source no Colab (GPU grátis).## 1. Instalar dependências

In [ ]:
!pip install -q churro-ocr[hf] transformers accelerate bitsandbytes!apt-get install -y tesseract-ocr tesseract-ocr-por > /dev/null 2>&1

## 2. Fazer upload das imagens do servidorNo servidor: `cd /home/pxtkhw/projetos/obitos && tar czf full_images.tar.gz output/full_images/`Depois faz upload do `full_images.tar.gz` aqui (3.7GB)

In [ ]:
from google.colab import filesimport osprint('Faz upload do full_images.tar.gz...')# uploaded = files.upload()  # Descomenta para fazer uploados.makedirs('full_images', exist_ok=True)!tar xzf full_images.tar.gz -C . 2>/dev/null || echo 'Faz upload primeiro!'print('Imagens extraídas')

## 3. Testar Churro-3B (modelo open-source, mais preciso que Gemini)Churro-3B: https://huggingface.co/stanford-oval/churro-3BCusto: ~15x menor que Gemini 2.5 Pro, precisão superior.

In [ ]:
import osos.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Usa GPUfrom churro_ocr import ChurroOCR# Teste com uma imagemimg_path = 'full_images/45283239.tiff'  # Imagem de testeif os.path.exists(img_path):    ocr = ChurroOCR(backend='hf', model='stanford-oval/churro-3B')    result = ocr.transcribe(img_path)    print('CHURRO OUTPUT:')    print(result.text[:500])else:    print('Imagem não encontrada. Faz upload primeiro.')

## 4. Alternativa: Transkribus model (Portuguese Handwriting)Modelos especializados: https://www.transkribus.org/models/Ex: AHP Handwritten Portuguese 19th-20th Centuries (CER 2.53%)

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModelfrom PIL import Imageimport torch# Carregar modelo TrOCR fine-tuned para portuguêsprocessor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')  # Substituir por modelo português se disponívelmodel = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')device = 'cuda' if torch.cuda.is_available() else 'cpu'model.to(device)def recognize_text(image_path):    image = Image.open(image_path).convert('RGB')    pixel_values = processor(image, return_tensors='pt').pixel_values.to(device)    generated_ids = model.generate(pixel_values)    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]    return generated_textif os.path.exists('full_images/45283239.tiff'):    text = recognize_text('full_images/45283239.tiff')    print('TrOCR OUTPUT:')    print(text[:500])

## 5. Processar todas as imagens (batch)Usa Churro-3B ou TrOCR para processar as 4152 imagens.

In [ ]:
import os, json, timeos.makedirs('htr_output', exist_ok=True)# Usar Churro se disponível, senão TrOCRuse_churro = Truetry:    from churro_ocr import ChurroOCR    ocr = ChurroOCR(backend='hf', model='stanford-oval/churro-3B')    print('Usando Churro-3B')except:    use_churro = False    print('Churro não disponível, usando TrOCR...')def process_image(img_path):    if use_churro:        result = ocr.transcribe(img_path)        return result.text    else:        # TrOCR fallback        image = Image.open(img_path).convert('RGB')        pixel_values = processor(image, return_tensors='pt').pixel_values.to(device)        generated_ids = model.generate(pixel_values)        return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]images = [f for f in os.listdir('full_images') if f.endswith('.tiff')]print(f'Total imagens: {len(images)}')for i, img in enumerate(sorted(images)[:10]):  # Começa com 10 para testar    try:        img_path = os.path.join('full_images', img)        text = process_image(img_path)        with open(f'htr_output/{os.path.splitext(img)[0]}.json', 'w') as f:            json.dump({'raw_text': text}, f)        print(f'[{i+1}] {img}: OK')        time.sleep(1)    except Exception as e:        print(f'[{i+1}] {img}: Error - {e}')print('Teste completo! Remove o [:10] para processar todas.')

## 6. Fazer download dos resultadosGera `htr_output.tar.gz` para fazer download.

In [ ]:
import tarfilefrom google.colab import fileswith tarfile.open('htr_output.tar.gz', 'w:gz') as tar:    tar.add('htr_output')files.download('htr_output.tar.gz')print('Download pronto!')